In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
os.chdir(r"C:\Users\DT1\Downloads")

In [16]:
img=cv2.imread("noisy_balloons.jpg")
cv2.imshow("original image", img)
cv2.waitKey(0)
cv2.destroyAllWindows()


In [24]:
font=cv2.FONT_HERSHEY_SIMPLEX
img=cv2.imread("history.jpg")
# balloon_image=cv2.imread("noisy_balloons.jpg")
 
cv2.imshow("original image", img)
blur=cv2.blur(img,(2,2))
cv2.putText(blur,"Blur",(10,30),font,1,(0,0,0),2)
# cv2.imshow("blurred image", blur)
median=cv2.medianBlur(img,3)
cv2.putText(median,"Median",(10,30),font,1,(0,0,0),2)
# cv2.imshow("bilateral blurred image", median)
# cv2.imshow("median blurred image", median)
gaussian=cv2.GaussianBlur(img,(3,3),0)
cv2.putText(gaussian,"Gaussian",(10,30),font,1,(255,255,255),2)
# cv2.imshow("gaussian blurred image", gaussian)
bilateral=cv2.bilateralFilter(img,9,75,75)
cv2.putText(bilateral,"Bilateral",(10,30),font,1,(0,0,0),2)
# cv2.imshow("bilateral image", bilateral)
# all above are low pass filters in one collection
blur_resized = cv2.resize(blur,(415,415))
median_resized = cv2.resize(median,(415,415))
gaussian_resized = cv2.resize(gaussian,(415,415))
bilateral_resized = cv2.resize(bilateral,(415,415))
collection=cv2.hconcat([blur_resized,median_resized,gaussian_resized,bilateral_resized])
cv2.imshow("collection", collection)
cv2.waitKey(0)
# cv2.imwrite("collection.png", collection)
cv2.destroyAllWindows()


# Image Pyramids

In [ ]:
img=cv2.imread("test_image.jpg")
lr_1=cv2.pyrDown(img)
lr_2=cv2.pyrDown(lr_1)
hr_1=cv2.pyrUp(lr_1)
cv2.imshow("Original",img)
cv2.imshow("LR_1",lr_1)
cv2.imshow("LR_2",lr_2)
cv2.imshow("HR_1",hr_1)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [3]:
img=cv2.imread("football.jpg")
layer=img.copy()
layers=[layer]
for i in range(6):
    layer=cv2.pyrDown(layer)
    layers.append(layer)
    cv2.imshow(str(i),layer)
cv2.imshow("Original",layer)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [5]:

img=cv2.imread("football.jpg")
layer=img.copy()
layers=[layer]
for i in range(6):
    layer=cv2.pyrDown(layer)
    layers.append(layer)
    # cv2.imshow(str(i),layer)
last_layer=layers[-1]
cv2.imshow("Last Layer",last_layer)
for i in range(5,0,-1):
    extend=cv2.pyrUp(layers[i],dstsize=(layers[i-1].shape[1],layers[i-1].shape[0]))
    laplacian=cv2.subtract(layers[i-1],extend)
    cv2.imshow(str(i),laplacian)
cv2.imshow("Original",img)
cv2.waitKey(0)
cv2.destroyAllWindows()

# Simple Image Smoothing/Blurring Techniques

In [4]:
import cv2
import numpy as np


# ============================================================
# 1. Load Image
# ============================================================

img = cv2.imread("test_image.jpg")

if img is None:
    print("Error: Could not load image.")
    exit()

# Resize for display
img = cv2.resize(img, (400, 300))


# ============================================================
# 2. Add Gaussian Noise
# ============================================================

def add_noise(image, amount):
    noise = np.random.normal(
        0,
        amount,
        image.shape
    )

    noisy = image.astype(np.float32) + noise

    noisy = np.clip(
        noisy,
        0,
        255
    ).astype(np.uint8)

    return noisy


# ============================================================
# 3. Add Title Above Image
# ============================================================

def add_title(image, title):

    height, width = image.shape[:2]

    # White header
    header = np.ones(
        (50, width, 3),
        dtype=np.uint8
    ) * 255

    cv2.putText(
        header,
        title,
        (10, 33),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 0, 0),
        2,
        cv2.LINE_AA
    )

    return cv2.vconcat([header, image])


# ============================================================
# 4. Trackbar Callbacks
# ============================================================

def nothing(x):
    pass


# ============================================================
# 5. Create Window
# ============================================================

cv2.namedWindow("Smoothing Analyzer")


# Kernel Size
cv2.createTrackbar(
    "Kernel",
    "Smoothing Analyzer",
    1,
    10,
    nothing
)

# Noise amount
cv2.createTrackbar(
    "Noise",
    "Smoothing Analyzer",
    20,
    100,
    nothing
)

# Gaussian Sigma
cv2.createTrackbar(
    "Sigma",
    "Smoothing Analyzer",
    10,
    100,
    nothing
)


# ============================================================
# 6. Main Loop
# ============================================================

while True:

    # --------------------------------------------------------
    # Get Trackbar Values
    # --------------------------------------------------------

    kernel_value = cv2.getTrackbarPos(
        "Kernel",
        "Smoothing Analyzer"
    )

    noise_value = cv2.getTrackbarPos(
        "Noise",
        "Smoothing Analyzer"
    )

    sigma_value = cv2.getTrackbarPos(
        "Sigma",
        "Smoothing Analyzer"
    )


    # --------------------------------------------------------
    # Make Kernel Odd
    # --------------------------------------------------------

    kernel = kernel_value * 2 + 1

    sigma = sigma_value / 10


    # --------------------------------------------------------
    # Generate Noisy Image
    # --------------------------------------------------------

    noisy = add_noise(
        img,
        noise_value
    )


    # --------------------------------------------------------
    # Average Blur
    # --------------------------------------------------------

    average = cv2.blur(
        noisy,
        (kernel, kernel)
    )


    # --------------------------------------------------------
    # Gaussian Blur
    # --------------------------------------------------------

    gaussian = cv2.GaussianBlur(
        noisy,
        (kernel, kernel),
        sigma
    )


    # --------------------------------------------------------
    # Median Blur
    # --------------------------------------------------------

    median = cv2.medianBlur(
        noisy,
        kernel
    )


    # --------------------------------------------------------
    # Bilateral Filter
    # --------------------------------------------------------

    bilateral = cv2.bilateralFilter(
        noisy,
        9,
        75,
        75
    )


    # ========================================================
    # Add Titles
    # ========================================================

    original_view = add_title(
        img,
        "Original"
    )

    noisy_view = add_title(
        noisy,
        f"Noisy ({noise_value})"
    )

    average_view = add_title(
        average,
        f"Average {kernel}x{kernel}"
    )

    gaussian_view = add_title(
        gaussian,
        f"Gaussian {kernel}x{kernel}"
    )

    median_view = add_title(
        median,
        f"Median {kernel}x{kernel}"
    )

    bilateral_view = add_title(
        bilateral,
        "Bilateral"
    )


    # ========================================================
    # First Row
    # ========================================================

    row1 = cv2.hconcat([
        original_view,
        noisy_view,
        average_view
    ])


    # ========================================================
    # Second Row
    # ========================================================

    row2 = cv2.hconcat([
        gaussian_view,
        median_view,
        bilateral_view
    ])


    # ========================================================
    # Final Dashboard
    # ========================================================

    dashboard = cv2.vconcat([
        row1,
        row2
    ])


    cv2.imshow(
        "Smoothing Analyzer",
        dashboard
    )


    # ========================================================
    # Keyboard Controls
    # ========================================================

    key = cv2.waitKey(30) & 0xFF

    if key == ord("q"):
        break

    elif key == ord("s"):

        cv2.imwrite(
            "smoothing_result.jpg",
            dashboard
        )

        print("Result saved as smoothing_result.jpg")


# ============================================================
# 7. Cleanup
# ============================================================

cv2.destroyAllWindows()
print("Analyzer closed. Goodbye!")

Analyzer closed. Goodbye!
